# 🛒 E-Commerce Intelligence Platform
## Phase 6 — Tableau Data Preparation
**Goal:** Prepare final clean file for Tableau Dashboard  
**Analyst:** Shivansh Pandey

In [1]:
# ============================================================
# CELL 1 — IMPORTS & UPLOAD
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
uploaded = files.upload()

print("✅ Files uploaded")

Saving churn_analysis.csv to churn_analysis.csv
Saving master_dataset.csv to master_dataset.csv
Saving rfm_segments.csv to rfm_segments.csv
✅ Files uploaded


In [2]:
# ============================================================
# CELL 2 — BUILD FINAL TABLEAU DATASET
# Merge all analysis results into one file
# ============================================================

# Load files
master = pd.read_csv('master_dataset.csv')
rfm    = pd.read_csv('rfm_segments.csv')
churn  = pd.read_csv('churn_analysis.csv')

# Convert dates
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])

# Add time columns
master['order_year']       = master['order_purchase_timestamp'].dt.year
master['order_month']      = master['order_purchase_timestamp'].dt.month
master['order_month_name'] = master['order_purchase_timestamp'].dt.strftime('%b')
master['order_yearmonth']  = master['order_purchase_timestamp'].dt.strftime('%Y-%m')
master['order_quarter']    = master['order_purchase_timestamp'].dt.quarter
master['order_dayofweek']  = master['order_purchase_timestamp'].dt.day_name()

# Merge RFM segments
master = master.merge(
    rfm[['customer_unique_id', 'segment',
         'recency', 'frequency', 'monetary',
         'r_score', 'f_score', 'm_score', 'total_score']],
    on='customer_unique_id', how='left'
)

# Merge churn labels
master = master.merge(
    churn[['customer_unique_id', 'is_churned', 'recency_days']],
    on='customer_unique_id', how='left'
)

# Clean category names
master['product_category_name_english'] = (
    master['product_category_name_english']
    .fillna('unknown')
    .str.replace('_', ' ')
    .str.title()
)

# Add revenue flag
master['revenue_band'] = pd.qcut(
    master['total_payment'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

print(f"✅ Tableau dataset built!")
print(f"   Shape   : {master.shape}")
print(f"   Columns : {list(master.columns)}")

✅ Tableau dataset built!
   Shape   : (110189, 46)
   Columns : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'is_late', 'days_diff', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'total_payment', 'payment_installments', 'payment_type', 'review_score', 'product_category_name_english', 'product_weight_g', 'seller_city', 'seller_state', 'order_year', 'order_month', 'order_month_name', 'order_yearmonth', 'order_quarter', 'order_dayofweek', 'segment', 'recency', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score', 'total_score', 'is_churned', 'recency_days', 'revenue_band']


In [3]:
# ============================================================
# CELL 3 — VERIFY & SAVE
# ============================================================

print("📊 Final Dataset Preview:")
print(master.head(3))

print(f"\n📋 Column Summary:")
print(master.dtypes)

print(f"\n✅ Missing Values:")
print(master.isnull().sum()[master.isnull().sum() > 0])

# Save final Tableau file
master.to_csv('tableau_final.csv', index=False)
files.download('tableau_final.csv')

print("\n✅ tableau_final.csv downloaded!")
print("   Move this file to data/final/ folder")

📊 Final Dataset Preview:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   

  order_estimated_delivery_date  delivery_days  is_late  ...  recency  \
0                    2017-10-18              8        0  ...      332   
1                    2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ tableau_final.csv downloaded!
   Move this file to data/final/ folder
